# 07 · Token accounting and cost

**Deck section 7** · slides 73–81

Cost is a design constraint, not an afterthought. In client work it is often the constraint
that decides the architecture — and the conversation goes very differently when you arrive
with a line-item table instead of a shrug.

**A note on the numbers.** The rates used here are the deck's illustrative ones: $3/MTok in,
$15/MTok out, cache reads at 0.1×. They are placeholders on purpose. Provider prices change
and model families differ; **the arithmetic is the transferable part**. Put a real rate card
into `costs.Rates(...)` before you quote anything to anybody.

**By the end you can**

- separate the four token categories on one request and say why one number for "tokens" makes
  all of them unoptimisable
- order a prompt by volatility, and break your own cache in one line to prove the rule
- price one grounded answer to the cent, and a month of them
- pull the cost levers in the right order, and name what each costs in quality


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import json
import numpy as np
import pandas as pd
import raglab
from raglab import viz, tables, catalog, context, costs, metrics, pipeline
viz.reset_figures("7."); tables.reset_tables("7.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
dev = [q for q in bundle.questions if q.slice == "dev"][:120]
print(f"{len(dev)} questions available for cost measurement")

---

## 7.1 Four categories on the same request

If your dashboard shows one number for tokens, you cannot optimise any of them — the three
that are cheap and the one that is not are all hidden inside it.


In [ ]:
viz.flow([
    ("Input", "new prompt tokens: instructions, query, retrieved context, tool definitions"),
    ("Output", "tokens generated: answer text, tool calls, structured output"),
    ("Cache write", "reusable prompt-prefix tokens processed and stored"),
    ("Cache read", "that prefix reused on a later request — the cheap one"),
], title="Four categories on the same request", kicker="Section 7 · token accounting",
   highlight=3,
   caption="Account for each separately. They have different prices, different levers, and "
           "three of them are under your control at design time.",
   source="Deck slide 74")

In [ ]:
# Measure the real split on this pipeline.
tr = pipe.run(dev[0].query, qid=dev[0].qid)
packed = tr._packed_obj
split = context.cacheable_prefix(packed)

rates = costs.Rates()
u_uncached = costs.Usage(input_tokens=packed.total_tokens,
                         output_tokens=tr.usage["output_tokens"])
u_cached = costs.Usage(input_tokens=split["volatile_tokens"],
                       output_tokens=tr.usage["output_tokens"],
                       cache_read=split["stable_tokens"])

tables.show(pd.DataFrame([
    ["Input (uncached)", f"{packed.total_tokens:,}", f"${rates.input_per_mtok:.2f}/MTok",
     f"${rates.cost(input_tokens=packed.total_tokens):.5f}"],
    ["— of which stable prefix", f"{split['stable_tokens']:,}",
     f"{split['stable_share']:.0%} of the prompt", "cacheable"],
    ["— of which volatile", f"{split['volatile_tokens']:,}",
     f"{1-split['stable_share']:.0%} of the prompt", "never cacheable"],
    ["Output", f"{tr.usage['output_tokens']:,}", f"${rates.output_per_mtok:.2f}/MTok",
     f"${rates.cost(output_tokens=tr.usage['output_tokens']):.5f}"],
    ["Cache read (if warm)", f"{split['stable_tokens']:,}",
     f"${rates.input_per_mtok * rates.cache_read_multiplier:.2f}/MTok",
     f"${rates.cost(cache_read=split['stable_tokens']):.5f}"],
], columns=["Category", "Tokens", "Rate", "Cost on this request"]),
    title="One request, split four ways",
    kicker="Measured on this pipeline",
    caption=f"Cold: ${u_uncached.cost(rates):.5f}. Warm: ${u_cached.cost(rates):.5f}. "
            f"The difference is one configuration decision, not an engineering project.",
    emphasize="Category")

---

## 7.2 The cache rule, and how to break it

Write once, read repeatedly. A prefix is a hit only if it is **byte-identical** to what was
written — change one character near the front and everything after it misses. That single
rule is the whole of cache engineering.


In [ ]:
viz.flow([("Stable prompt prefix", "system prompt · tools · few-shot exemplars"),
          ("Cache write", "1.25× base input, once"),
          ("Cached prefix", "held for the retention window"),
          ("Cache read", "0.1× base input, on every later request that shares it")],
         title="Cache lifecycle: write once, read repeatedly",
         kicker="Section 7 · caching", highlight=3,
         caption="Cache efficiency depends on keeping the reusable prefix stable. Changing "
                 "earlier prompt content invalidates every later hit.",
         source="Deck slide 75")

In [ ]:
def simulate(prefix_fn, n=60, label=""):
    '''Run n requests through a prefix cache and report what it cost.'''
    cache = costs.PromptCache()
    total = costs.Usage()
    for i, q in enumerate(dev[:n]):
        t = pipe.run(q.query, qid=q.qid, acl_groups=bundle.personas.get(q.persona))
        p = t._packed_obj
        s = context.cacheable_prefix(p)
        prefix_text = prefix_fn(i, q)
        total = total + cache.lookup(prefix_text, s["stable_tokens"])
        total = total + costs.Usage(input_tokens=s["volatile_tokens"],
                                    output_tokens=t.usage["output_tokens"])
    return {"label": label, **cache.report(), "cost": total.cost(rates),
            "cache_read": total.cache_read, "cache_write": total.cache_write}


import datetime, uuid

runs = [
    simulate(lambda i, q: context.SYSTEM_CONTRACT, label="stable prefix (correct)"),
    simulate(lambda i, q: f"Today is 2026-08-31 14:{i:02d}:{i:02d}\n" + context.SYSTEM_CONTRACT,
             label="timestamp at the front"),
    simulate(lambda i, q: f"request-id: {uuid.uuid5(uuid.NAMESPACE_URL, str(i)).hex}\n"
                          + context.SYSTEM_CONTRACT,
             label="request id at the front"),
    simulate(lambda i, q: json.dumps({"tools": ["search", "fetch"], "v": 1},
                                     sort_keys=(i % 2 == 0)) + context.SYSTEM_CONTRACT,
             label="tool schema serialised unsorted"),
    simulate(lambda i, q: context.SYSTEM_CONTRACT + ("\nVariant B." if i % 2 else ""),
             label="A/B testing the system prompt"),
]

frame = pd.DataFrame(runs)[["label", "requests", "hits", "hit_rate", "distinct_prefixes",
                            "cost"]]
frame["cost"] = frame["cost"].map(lambda v: f"${v:.4f}")
frame["vs correct"] = [f"{(r['cost']/runs[0]['cost']-1):+.1%}" for r in runs]
tables.show(frame, title="Four ways to destroy a prompt cache, priced",
            kicker="Cache killers, measured",
            caption="Every row runs the identical 60 questions through the identical "
                    "pipeline. The only thing that changed is what sits at the front of the "
                    "prompt.",
            emphasize="hit_rate",
            highlight_rows=lambda r: r["label"] == "stable prefix (correct)")

In [ ]:
tables.show(pd.DataFrame(context.CACHE_KILLERS,
                         columns=["Cache killer", "Why it kills the cache", "The fix"]),
            title="The four, written down", kicker="Prompt hygiene",
            caption="Track cache hit rate as a first-class metric next to p95 latency. A hit "
                    "rate that drops overnight usually means someone edited the top of the "
                    "prompt.",
            source="Deck slide 79", emphasize="Cache killer")

### Order the prompt by how often each part changes

This is a five-minute change that often halves input spend, and it costs nothing in quality.


In [ ]:
tables.show(pd.DataFrame([[name, changes, place] for name, changes, place
                          in context.PROMPT_VOLATILITY],
                         columns=["Prompt segment", "Changes", "Place it"]),
            title="Order by volatility: stable content first, volatile content last",
            kicker="Cache-aware layout",
            caption="`context.build_prompt()` emits exactly this order. The question goes "
                    "last, and a short restatement after the evidence also happens to sit in "
                    "the strong attention position (notebook 05).",
            source="Deck slide 79", emphasize="Prompt segment")

---

## 7.3 Two providers, and the comparison you should refuse to make


In [ ]:
catalog.PROVIDER_CACHE.show()

In [ ]:
# The retention decision, as arithmetic rather than preference.
prefix_tokens = split["stable_tokens"]
rows = []
for reuse in (2, 5, 10, 25, 60, 200):
    for label, r in (("5-minute write (1.25×)", costs.ANTHROPIC_5M),
                     ("1-hour write (2.0×)", costs.ANTHROPIC_1H)):
        write_cost = r.cost(cache_write=prefix_tokens)
        read_cost = r.cost(cache_read=prefix_tokens) * (reuse - 1)
        uncached = r.cost(input_tokens=prefix_tokens) * reuse
        rows.append([label, reuse, round(write_cost + read_cost, 5), round(uncached, 5),
                     f"{(write_cost + read_cost) / uncached - 1:+.0%}"])

frame = pd.DataFrame(rows, columns=["Retention", "Reuses inside the window", "Cached total",
                                    "Uncached total", "Saving"])
tables.show(frame.pivot(index="Reuses inside the window", columns="Retention",
                        values="Saving").reset_index(),
            title=f"Does the write pay for itself? ({prefix_tokens:,}-token prefix)",
            kicker="Retention arithmetic",
            caption="Choose the retention by how often the prefix is reused inside the "
                    "window, not by which number looks smaller. At two reuses a 1-hour write "
                    "does not pay for itself; at sixty it barely matters which you picked.",
            emphasize="Reuses inside the window")

tables.callout(
    "<b>Do not compare cache costs across providers without fixing the model, the retention "
    "policy, the prompt shape and the expected reuse rate.</b> Four variables, and a "
    "comparison that moves any of them is measuring something other than the cache. When a "
    "client asks which provider is cheaper, the honest answer is “for what shape of traffic?” "
    "— and then you build the table above with their numbers.", kind="cost")

---

## 7.4 What one grounded answer actually costs

The deck's worked example, reproduced exactly, then rebuilt from *our* measured token counts
so you can see the difference between a slide and a system.


In [ ]:
deck = costs.unit_economics(prefix_tokens=3000, k=8, tokens_per_chunk=550,
                            question_tokens=200, output_tokens=450, rerank_candidates=50,
                            cached=True, monthly_queries=200_000)

lines = []
for name, toks, rate, cost in deck["lines"]:
    lines.append([name, f"{toks:,}" if toks else "—",
                  f"${rate:.2f}" if rate else "—", f"${cost:.4f}"])
lines.append(["Total per answered query", "", "", f"≈ ${deck['total_per_query']:.4f}"])
tables.show(pd.DataFrame(lines, columns=["Line item", "Tokens", "Rate", "Cost"]),
            title="What one grounded answer costs",
            kicker=f"Worked example · illustrative rates · "
                   f"${rates.input_per_mtok:.0f}/MTok in, ${rates.output_per_mtok:.0f}/MTok out",
            caption=f"At {deck['monthly_queries']:,} queries/month: "
                    f"${deck['monthly']:,.0f}. Without prompt caching on the prefix, the same "
                    f"volume is about "
                    f"${costs.unit_economics(cached=False)['monthly']:,.0f} — caching removes "
                    f"roughly a quarter of the bill for one afternoon of work.",
            source="Deck slide 80", emphasize="Line item")

In [ ]:
# Now the same table from what this pipeline actually spends.
rs = pipeline.evaluate(pipe, dev, pipe.chunks, personas=bundle.personas)
s = metrics.summarize(rs)
measured = costs.unit_economics(
    prefix_tokens=split["stable_tokens"],
    k=pipe.cfg.k,
    tokens_per_chunk=int(np.mean([b["tokens"] for b in tr.packed])),
    question_tokens=int(packed.tokens["question"]),
    output_tokens=int(s["tokens_out"]),
    rerank_candidates=pipe.cfg.rerank_depth, cached=True)

tables.show(pd.DataFrame([
    ["Cached prefix", 3000, split["stable_tokens"]],
    ["Evidence tokens (k × per-chunk)", 8 * 550,
     pipe.cfg.k * int(np.mean([b["tokens"] for b in tr.packed]))],
    ["Question + state", 200, int(packed.tokens["question"])],
    ["Generated answer", 450, int(s["tokens_out"])],
    ["Cost per answered query", round(deck["total_per_query"], 4),
     round(measured["total_per_query"], 4)],
    ["At 200k queries/month", round(deck["monthly"]), round(measured["monthly"])],
], columns=["Line", "Deck's illustration", "This pipeline, measured"]),
    title="The slide versus the system",
    kicker="Calibrating the model against reality",
    caption="Our chunks are smaller and our reader is terser, so our per-query cost is lower "
            "and our evidence budget is under-spent. Both numbers are right; only one of them "
            "is about your system.",
    emphasize="This pipeline, measured")

In [ ]:
# Index-side economics: the one-off you compare the query-side saving against.
from raglab import chunking

corpus_tokens = sum(chunking.approx_tokens(d.body) for d in bundle.documents)
scaled = 2_000_000 * 400          # the deck's 2M chunks × 400 tokens
embed_cost = scaled * rates.embed_per_mtok / 1e6
ctx_enrich = rates.cost(cache_read=scaled * 0.8, input_tokens=scaled * 0.2,
                        output_tokens=scaled * 0.15)

tables.show(pd.DataFrame([
    ["This corpus, as built", f"{corpus_tokens:,} tokens",
     f"${corpus_tokens * rates.embed_per_mtok / 1e6:.4f}",
     "embedding the whole thing costs less than one query"],
    ["A client corpus, 2M chunks × 400 tok", f"{scaled:,} tokens", f"${embed_cost:,.0f}",
     "embedding is on the order of tens of dollars"],
    ["…plus a contextual-enrichment pass", f"{scaled:,} tokens read, ~15% generated",
     f"${ctx_enrich:,.0f}",
     "affordable only because the parent document sits in a cached prefix"],
    ["Query-side saving from k=8 → k=6", "at 200k queries/month",
     f"${(costs.unit_economics(k=8)['monthly'] - costs.unit_economics(k=6)['monthly']):,.0f}"
     "/month",
     "recurring, and it costs full-chain recall"],
], columns=["Spend", "Volume", "Cost", "Reading"]),
    title="Index-side is a one-off; query-side is forever",
    kicker="Where to spend the next dollar",
    caption="Compare the one-off against the recurring saving and index-time work almost "
            "always wins. That is the deck's second closing sentence, priced.",
    emphasize="Cost")

---

## 7.5 The cost levers, in the order you should pull them

Work top down. The first three are free in quality terms; the last two are trades you must
declare out loud.


In [ ]:
catalog.COST_LEVERS.show()

In [ ]:
# Measure the levers we can actually pull here.
base = metrics.summarize(rs)
base_cost = base["cost_usd"]

lever_rows = []

# 1 · caching the stable prefix
warm = runs[0]["cost"] / 60
cold = costs.Usage(input_tokens=packed.total_tokens,
                   output_tokens=int(base["tokens_out"])).cost(rates)
lever_rows.append(["1 · Cache the stable prefix", f"{(warm - cold)/cold:+.1%}",
                   "0.000", "Nothing — the prompt is byte-identical either way"])

# 2 · dedup
nodedup = pipeline.evaluate(pipe.variant("nodedup", dedup=False), dev, pipe.chunks,
                            personas=bundle.personas)
s2 = metrics.summarize(nodedup)
lever_rows.append(["2 · Deduplicate near-identical chunks",
                   f"{(base['cost_usd'] - s2['cost_usd'])/s2['cost_usd']:+.1%}",
                   f"{base['full_chain_recall'] - s2['full_chain_recall']:+.3f}",
                   "Usually improves quality — duplicates are distractors that also cost "
                   "tokens"])

# 3 · cap output
terse = pipe.variant("terse")
from raglab import generate
terse.generator = generate.ExtractiveGenerator(max_sentences=1)
s3 = metrics.summarize(pipeline.evaluate(terse, dev, pipe.chunks, personas=bundle.personas))
lever_rows.append(["3 · Cap output length / terse schema",
                   f"{(s3['cost_usd'] - base_cost)/base_cost:+.1%}",
                   f"{s3['full_chain_recall'] - base['full_chain_recall']:+.3f}",
                   "None if the output contract is well specified"])

# 4 · lower k
for k in (6, 4):
    sk = metrics.summarize(pipeline.evaluate(pipe.variant(f"k{k}", k=k), dev, pipe.chunks,
                                             personas=bundle.personas))
    lever_rows.append([f"4 · Lower k to {k} (from {pipe.cfg.k})",
                       f"{(sk['cost_usd'] - base_cost)/base_cost:+.1%}",
                       f"{sk['full_chain_recall'] - base['full_chain_recall']:+.3f}",
                       "Real risk on multi-hop — measure the tail, not the mean"])

# 7 · drop the reranker
norr = metrics.summarize(pipeline.evaluate(pipe.variant("norr", rerank="none"), dev,
                                           pipe.chunks, personas=bundle.personas))
lever_rows.append(["7 · Drop the reranker",
                   f"{(norr['cost_usd'] - base_cost)/base_cost:+.1%}",
                   f"{norr['full_chain_recall'] - base['full_chain_recall']:+.3f}",
                   "The worst trade on the list — large quality loss, trivial saving"])

tables.show(pd.DataFrame(lever_rows, columns=[
    "Lever", "Cost change", "Full-chain recall change", "What the deck says it costs"]),
    title="The levers, pulled and measured on this system",
    kicker="Measured",
    caption="Note lever 7. The saving is inside the rounding and the quality cost is not — "
            "which is exactly why it sits at the bottom of the list.",
    emphasize="Cost change",
    highlight_rows=lambda r: r["Lever"].startswith("7"))

In [ ]:
viz.scatter_frontier(
    [("baseline k=8", base_cost * 1e4, base["full_chain_recall"])] +
    [(f"k={k}", metrics.summarize(pipeline.evaluate(pipe.variant(f"f{k}", k=k), dev,
                                                    pipe.chunks,
                                                    personas=bundle.personas))["cost_usd"] * 1e4,
      metrics.summarize(pipeline.evaluate(pipe.variant(f"f{k}", k=k), dev, pipe.chunks,
                                          personas=bundle.personas))["full_chain_recall"])
     for k in (3, 5, 12)] +
    [("no reranker", norr["cost_usd"] * 1e4, norr["full_chain_recall"])],
    xlabel="cost per query (× $10⁻⁴)", ylabel="full-chain recall",
    title="The cost/quality frontier, with the operating point marked",
    kicker="Frontier", chosen="baseline k=8",
    caption="“A stated cost/quality frontier with the chosen operating point marked” is the "
            "build rubric's bar for exceeding on cost and latency. Bring this, not an "
            "assertion that the system is efficient.")

---

## 7.6 Interview Q5 — saying no well

> *"Agentic search costs $0.90 on hard questions. Finance wants $0.15. What do you change, and
> what do you refuse to change?"*

The panel is testing three things: whether you can decompose a per-query cost into line items
from memory, whether you optimise the *distribution* rather than the worst case, and whether
you will push back with a quantified consequence instead of silently degrading quality.

The first move is always the same — **reframe from worst case to blended cost**.


In [ ]:
hard_share = 0.08
worst_case = 0.90
single_shot = measured["total_per_query"]

blended_now = hard_share * worst_case + (1 - hard_share) * single_shot
print(f"if {hard_share:.0%} of queries are hard:")
print(f"  worst case                 ${worst_case:.2f}")
print(f"  single-shot                ${single_shot:.4f}")
print(f"  blended                    ${blended_now:.4f}")
print(f"  finance's target           $0.15")
print(f"  → {'already inside target' if blended_now <= 0.15 else 'over target'}\n")

scenarios = []
for share in (0.05, 0.08, 0.15, 0.30, 0.50):
    b = share * worst_case + (1 - share) * single_shot
    scenarios.append([f"{share:.0%}", f"${b:.4f}",
                      "inside target" if b <= 0.15 else f"over by ${b-0.15:.3f}"])
tables.show(pd.DataFrame(scenarios, columns=["Share of queries that are hard", "Blended cost",
                                             "Against a $0.15 target"]),
            title="Get the distribution before you redesign anything",
            kicker="Reframing the question",
            caption="The first question is not “how do we make hard queries cheaper”. It is "
                    "“what fraction of traffic is hard”. Often the blended number is already "
                    "near target and the whole exercise was about a number nobody had "
                    "computed.",
            emphasize="Blended cost")

In [ ]:
tables.show(pd.DataFrame([
    ["Escalate, do not loop by default",
     "Single-shot first; enter the loop only when the sufficiency check fails",
     "Removes most of the multiplier, because most traffic is not hard",
     "Notebook 08 measures it"],
    ["Cache the stable prefix and tool schemas", "Prompt hygiene, once",
     f"{abs((warm-cold)/cold):.0%} of input spend on this pipeline", "Nothing"],
    ["Carry a compacted evidence summary between turns",
     "Instead of the full text of every prior turn",
     "Turns the loop's token growth from quadratic to roughly linear",
     "Some risk of dropping a detail — summarise, do not truncate"],
    ["Cap turns", "A hard maximum, typically 4–8", "Bounds the tail, which is what hurts",
     "Costs full-chain recall on the hardest questions — quantify it"],
    ["Small model for decomposition and sufficiency",
     "Large model only for synthesis", "30–60% on the loop's overhead calls",
     "A router you must also evaluate — a second system"],
], columns=["Lever", "What it is", "What it buys", "What it costs"]),
    title="Cheap wins, in order",
    kicker="The plan", emphasize="Lever")

tables.callout(
    "<b>What I refuse:</b> removing the grounding and abstention checks, and removing the "
    "trace. Both are cheap and both are what stop a wrong answer from becoming an incident."
    "<br><br><b>And quantify the residual rather than agreeing to the number:</b> “I can reach "
    "$0.22 blended without quality loss. Getting to $0.15 means capping at two turns, which "
    "costs roughly X points of full-chain recall on multi-hop. That is a business decision — "
    "here is the number.”"
    "<br><br>A candidate who will not name something they refuse to trade is not senior yet. "
    "That is the sentence the panel is waiting for.",
    kind="interview", title="The part most candidates skip")

---

## 7.7 Checkpoint

1. Your cache hit rate dropped from 0.94 to 0.11 overnight and nobody deployed. What
   happened?
2. A client wants a 40% cost reduction. Which three levers do you pull first, and what do you
   tell them each costs?
3. You are asked whether prompt caching is worth it for a workload with 1.5 average reuses per
   prefix. Answer with arithmetic.


In [ ]:
print("1 ·  Something changed at the FRONT of the prompt. In order of likelihood: a timestamp")
print("     or request id crept into the system prompt; a tool schema started serialising")
print("     unsorted; someone began A/B testing the prompt per request; or a tenant config")
print("     moved above the stable block. §7.2 priced all four. Check the prompt diff before")
print("     you check the provider's status page.\n")

print("2 ·  Cache the prefix, deduplicate before packing, cap output length — in that order.")
print("     Measured on this pipeline:")
for r in lever_rows[:3]:
    print(f"       {r[0]:<40} cost {r[1]:>7}   quality {r[2]:>7}")
print("     All three are free in quality terms. If 40% is still not reached, the next lever")
print("     is lowering k, and THAT one you present as a trade with a number attached.\n")

r5 = costs.ANTHROPIC_5M
w = r5.cost(cache_write=prefix_tokens)
rd = r5.cost(cache_read=prefix_tokens) * 0.5
unc = r5.cost(input_tokens=prefix_tokens) * 1.5
print("3 ·  At 1.5 reuses, with a 1.25x write and a 0.1x read:")
print(f"       cached   write {w:.6f} + 0.5 reads {rd:.6f} = ${w+rd:.6f}")
print(f"       uncached 1.5 × full input                    = ${unc:.6f}")
print(f"       → caching is {'cheaper' if w+rd < unc else 'MORE EXPENSIVE'} "
      f"({(w+rd)/unc-1:+.0%})")
print("     Below roughly two reuses inside the retention window the write does not pay for")
print("     itself. Monitor cached and cache-write token counters and measure it; whether")
print("     reuse offsets write cost is an empirical question per workload, not a rule.")

---

## What carries forward

- Four categories, four different prices. One number for "tokens" makes all four
  unoptimisable.
- The cache rule is byte-identity of the prefix. Order the prompt by volatility and track hit
  rate next to p95.
- Price one answer, then a month of them, with a real rate card. It turns "AI is expensive"
  into a line-item conversation the client can own.
- Pull the free levers first, and declare the trades out loud with a number attached — 
  including the one you refuse to make.

**Next:** `08_agentic_search_and_evaluation.ipynb` — decompose, choose a tool, retrieve, check
sufficiency, stop; and score the trace rather than the answer.
